In [ ]:

import torch
import numpy as np
from accelerate import Accelerator
from transformers import Sam3VideoModel, Sam3VideoProcessor
import cv2
from torchcodec.decoders import VideoDecoder  
from tqdm import tqdm 

# ---------------------------
# Helper: iterate frames with TorchCodec VideoDecoder
# ---------------------------
def iter_video_frames_torchcodec(
    path: str,
    *,
    device: str = "cpu",          # TorchCodec decode device ("cpu" or "cuda")
    max_frames: int | None = None,
    stride: int = 1,
    to_numpy: bool = True,        # Convert CHW torch.uint8 → HWC np.uint8 RGB
):
    """
    Yields (frame_index, frame_hwc_uint8) using TorchCodec's VideoDecoder.
    - frame is RGB uint8 [H,W,3]
    - stride: take every `stride`-th frame
    - max_frames: stop after N yielded frames (None = all)
    """
    dec = VideoDecoder(path, device=device)  # uses system FFmpeg under the hood
    total = dec.metadata.num_frames
    if not max_frames:
        max_frames == total
    yielded = 0
    # for i in range(0, total, stride):
    for i in tqdm(range(0, total, stride)):
        frame_chw = dec[i]  # torch.uint8 [C,H,W], RGB order in TorchCodec
        if to_numpy:
            # Convert to HWC numpy (RGB)
            frame_hwc = frame_chw.permute(1, 2, 0).contiguous().cpu().numpy()
        else:
            frame_hwc = frame_chw  # keep as CHW torch.uint8 if preferred
        yield i, frame_hwc
        yielded += 1
        if max_frames is not None and yielded >= max_frames:
            break


In [ ]:
# ---------------------------
# Configure model + processor (Transformers SAM-3)
# ---------------------------
accel  = Accelerator()
device = accel.device                # e.g., cuda:0
dtype  = torch.bfloat16

model = Sam3VideoModel.from_pretrained("facebook/sam3").to(device, dtype=dtype)
processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")

# ---------------------------
# Start a *streaming* video session (no video passed → no preload)
# ---------------------------
inference_session = processor.init_video_session(
    inference_device=device,
    inference_state_device="cpu",   # keep tracker state on CPU to reduce VRAM
    processing_device="cpu",
    video_storage_device="cpu",
    dtype=dtype,
)

# ---------------------------
# Add TEXT-BASED PCS prompts (one concept per call)
# ---------------------------
TEXT_PROMPTS = ["chicken", "bird"]  # edit as needed
for phrase in TEXT_PROMPTS:
    inference_session = processor.add_text_prompt(
        inference_session=inference_session,
        text=phrase,
    )

# ---------------------------
# Stream frames from TorchCodec and process one-by-one
# ---------------------------
VIDEO = "../data/test.mp4"  # change to your path

Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

In [3]:
processed_masks = {}
for frame_idx, frame in iter_video_frames_torchcodec(
    VIDEO, device="cpu", stride=1, to_numpy=True
):
    # Prepare a single RGB frame (H,W,3) → processor tensor on 'device'
    inputs = processor(images=frame, device=device, return_tensors="pt")

    # Push this frame to the model (streaming step)
    sam3_out = model(
        inference_session=inference_session,
        frame=inputs.pixel_values[0],     # per-frame tensor
        reverse=False,                    # forward in time
    )

    # Post-process back to original resolution and unified dict
    # (masks, boxes, scores, object_ids when available)
    outputs = processor.postprocess_outputs(
        inference_session,
        sam3_out,
        original_sizes=inputs.original_sizes,
    )

    # # Inspect masks if present
    # masks = outputs.get("masks", None)       # Tensor [N, H, W] at original video resolution
    # boxes = outputs.get("boxes", None)       # Tensor [N, 4] XYXY pixels
    # ids   = outputs.get("object_ids", None)  # Tensor [N] (stable track IDs)
    # if masks is not None:
    #     print(f"Frame {frame_idx}: masks {tuple(masks.shape)} "
    #           f"| boxes {tuple(boxes.shape) if boxes is not None else '—'} "
    #           f"| obj_ids {tuple(ids.shape) if ids is not None else '—'}")
    # else:
    #     print(f"Frame {frame_idx}: no masks (keys={list(outputs.keys())})")
    processed_masks[frame_idx] = outputs

  2%|▏         | 16/733 [00:02<01:49,  6.55it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

100%|█████████▉| 732/733 [03:45<00:00,  3.25it/s]


In [ ]:
processed_masks

{0: {'object_ids': tensor([], dtype=torch.int64),
  'scores': tensor([]),
  'boxes': tensor([], size=(0, 4)),
  'masks': tensor([], size=(0, 576, 704), dtype=torch.bool),
  'prompt_to_obj_ids': {}},
 1: {'object_ids': tensor([], dtype=torch.int64),
  'scores': tensor([]),
  'boxes': tensor([], size=(0, 4)),
  'masks': tensor([], size=(0, 576, 704), dtype=torch.bool),
  'prompt_to_obj_ids': {}},
 2: {'object_ids': tensor([], dtype=torch.int64),
  'scores': tensor([]),
  'boxes': tensor([], size=(0, 4)),
  'masks': tensor([], size=(0, 576, 704), dtype=torch.bool),
  'prompt_to_obj_ids': {}},
 3: {'object_ids': tensor([], dtype=torch.int64),
  'scores': tensor([]),
  'boxes': tensor([], size=(0, 4)),
  'masks': tensor([], size=(0, 576, 704), dtype=torch.bool),
  'prompt_to_obj_ids': {}},
 4: {'object_ids': tensor([], dtype=torch.int64),
  'scores': tensor([]),
  'boxes': tensor([], size=(0, 4)),
  'masks': tensor([], size=(0, 576, 704), dtype=torch.bool),
  'prompt_to_obj_ids': {}},
 5: {

: 

In [ ]:
processed_masks[0]